In [16]:
!pip install supabase python-dotenv pandas numpy scipy PySastrawi


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# =========================================================
# CELL 1 - IMPORT LIBRARY + KONEKSI SUPABASE
# =========================================================
import os
import re
import math
import json
import pickle
import sys
import scipy.sparse as sp
import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from supabase import create_client
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

BACKEND_DIR = next(
    path for path in (Path.cwd(), Path.cwd().parent, Path.cwd() / "backend")
    if (path / "src" / "preprocessing").is_dir()
)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from src.preprocessing.stopwords import get_stopwords

load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

SOURCE_TABLE = "cleaned_papers_results"
TFIDF_TABLE = "tfidf_weights"

print("✅ Supabase client siap")

✅ Supabase client siap


In [18]:
# =========================================================
# CELL 2 - LOAD DATA DARI CLEANED PAPERS
# =========================================================
print("📥 Mengambil data dari Supabase...")

all_data = []
batch_size = 1000
offset = 0

while True:
    response = (
        supabase.table(SOURCE_TABLE)
        .select("id, title, abstract,category")
        .range(offset, offset + batch_size - 1)
        .execute()
    )

    batch = response.data or []

    if not batch:
        break

    all_data.extend(batch)
    print(f"  → Batch {offset // batch_size + 1}: {len(batch)} data")

    if len(batch) < batch_size:
        break

    offset += batch_size

df = pd.DataFrame(all_data)

if df.empty:
    raise ValueError("❌ Data kosong. Cek tabel cleaned_papers_results.")

df["id"] = df["id"].astype("int64")
df["title"] = df["title"].fillna("").astype(str)
df["abstract"] = df["abstract"].fillna("").astype(str)
df["category"] = df["category"].fillna("unknown").astype(str)

print(f"✅ Total artikel: {len(df)}")
print("Distribusi kategori:")
print(df["category"].value_counts(dropna=False))
df.head(3)

📥 Mengambil data dari Supabase...
  → Batch 1: 200 data
✅ Total artikel: 200
Distribusi kategori:
category
machine learning      50
mobile application    50
web application       50
cyber security        50
Name: count, dtype: int64


,id,title,abstract,category
0,9,Open-environment machine learning,… open learning or open ML for short. Note tha...,machine learning
1,5,A guide to machine learning for biologists,… A machine learning task is an objective spec...,machine learning
2,33,Financial applications of machine learning: A ...,… of machine learning and deep learning in … o...,machine learning


In [19]:
# =========================================================
# CELL 3 - STOPWORDS + STEMMER
# =========================================================
stop_words = get_stopwords()

stemmer = StemmerFactory().create_stemmer()

print("✅ Stopwords dan stemmer siap")

✅ Stopwords dan stemmer siap


In [20]:
# =========================================================
# CELL 4 - FUNGSI PREPROCESSING TITLE + ABSTRACT
# =========================================================
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess_to_tokens(text):
    cleaned = clean_text(text)
    tokens = cleaned.split()
    tokens = [token for token in tokens if token not in stop_words and len(token) > 1]
    tokens = [stemmer.stem(token) for token in tokens]
    return tokens

print("✅ Fungsi preprocessing siap")

✅ Fungsi preprocessing siap


In [21]:
# =========================================================
# CELL 5 - MEMBUAT DOKUMEN VSM: TITLE + ABSTRACT
# =========================================================
df["title_tokens"] = df["title"].apply(preprocess_to_tokens)
df["abstract_tokens"] = df["abstract"].apply(preprocess_to_tokens)

# Sesuai proposal: judul dan abstrak digabung menjadi satu dokumen
df["document_tokens"] = df["title_tokens"] + df["abstract_tokens"]
df["document_text"] = df["document_tokens"].apply(lambda tokens: " ".join(tokens))

print("✅ Dokumen VSM dari title + abstract selesai dibuat")
print("Dokumen kosong:", int(df["document_text"].eq("").sum()))

df[["id", "category", "title", "document_text"]].head(5)

✅ Dokumen VSM dari title + abstract selesai dibuat
Dokumen kosong: 0


,id,category,title,document_text
0,9,machine learning,Open-environment machine learning,open environment machine learning open learnin...
1,5,machine learning,A guide to machine learning for biologists,guide machine learning biologists machine lear...
2,33,machine learning,Financial applications of machine learning: A ...,financial applications machine learning litera...
3,39,machine learning,Applying machine learning to study fluid mecha...,applying machine learning study fluid mechanic...
4,52,mobile application,A survey of performance optimization for mobil...,survey performance optimization mobile applica...


In [22]:
# =========================================================
# CELL 6 - AUDIT DOKUMEN TF-IDF
# =========================================================
df["len_title_tokens"] = df["title_tokens"].apply(len)
df["len_abstract_tokens"] = df["abstract_tokens"].apply(len)
df["len_document_tokens"] = df["document_tokens"].apply(len)

print("Rata-rata token title:", round(df["len_title_tokens"].mean(), 2))
print("Rata-rata token abstract:", round(df["len_abstract_tokens"].mean(), 2))
print("Rata-rata token gabungan:", round(df["len_document_tokens"].mean(), 2))
print("Dokumen kosong:", int((df["len_document_tokens"] == 0).sum()))

df[["category", "title", "title_tokens", "abstract_tokens", "document_tokens"]].sample(
    min(5, len(df)),
    random_state=42
)

Rata-rata token title: 8.28
Rata-rata token abstract: 19.57
Rata-rata token gabungan: 27.84
Dokumen kosong: 0


,category,title,title_tokens,abstract_tokens,document_tokens
95,web application,ATON: An open-source framework for creating im...,"[aton, open, source, framework, creating, imme...","[web, browsers, available, virtually, all, com...","[aton, open, source, framework, creating, imme..."
15,machine learning,Human-in-the-loop machine learning: a state of...,"[human, loop, machine, learning, state, art]","[researchers, defining, new, types, interactio...","[human, loop, machine, learning, state, art, r..."
30,mobile application,Features and Functionalities of Medical Mobile...,"[features, functionalities, medical, mobile, a...","[mobile, applications, aid, users, choosing, s...","[features, functionalities, medical, mobile, a..."
158,mobile application,Gamification in personal health management: a ...,"[gamification, personal, health, management, f...","[review, article, explores, concept, applicati...","[gamification, personal, health, management, f..."
128,machine learning,"Impact of machine learning on management, heal...","[impact, machine, learning, management, health...","[machine, learning, deep, learning, two, most,...","[impact, machine, learning, management, health..."


In [23]:
# =========================================================
# CELL 7 - HITUNG DF DAN IDF
# Rumus proposal:
# IDF = log10(N / df_t)
# =========================================================
N = len(df)

document_frequency = Counter()

for tokens in df["document_tokens"]:
    unique_terms = set(tokens)
    document_frequency.update(unique_terms)

idf_scores = {
    term: math.log10(N / df_value)
    for term, df_value in document_frequency.items()
    if df_value > 0
}

terms = sorted(idf_scores.keys())

print("✅ IDF selesai dihitung")
print("Jumlah dokumen:", N)
print("Jumlah term:", len(terms))

pd.DataFrame([
    {
        "term": term,
        "df": document_frequency[term],
        "idf": idf_scores[term]
    }
    for term in terms[:10]
])

✅ IDF selesai dihitung
Jumlah dokumen: 200
Jumlah term: 1723


,term,df,idf
0,000,1,2.301030
1,10,3,1.823909
2,100m,1,2.301030
3,12,1,2.301030
4,13,1,2.301030
5,16,1,2.301030
6,19,7,1.455932
7,1959,1,2.301030
8,1k,1,2.301030
9,20,1,2.301030


In [24]:
# =========================================================
# CELL 8 - HITUNG TF-IDF
# Rumus:
# TF = jumlah kemunculan term / total kata dokumen
# TF-IDF = TF × IDF
# =========================================================
tfidf_records = []
tfidf_detail_records = []

ts = datetime.now(timezone.utc).isoformat()

for _, row in df.iterrows():
    doc_id = int(row["id"])
    title = row["title"]
    category = row["category"]
    tokens = row["document_tokens"]
    total_terms = len(tokens)

    if total_terms == 0:
        continue

    term_counts = Counter(tokens)

    for term, count in term_counts.items():
        tf = count / total_terms
        idf = idf_scores.get(term, 0)
        tfidf_score = tf * idf

        tfidf_records.append({
            "doc_id": doc_id,
            "category": category,
            "title": title,
            "term": term,
            "tfidf_score": float(tfidf_score),
            "updated_at": ts
        })

        tfidf_detail_records.append({
            "doc_id": doc_id,
            "category": category,   
            "title": title,
            "term": term,
            "term_count": count,
            "total_terms": total_terms,
            "tf": tf,
            "df": document_frequency[term],
            "idf": idf,
            "tfidf_score": tfidf_score
        })

tfidf_df = pd.DataFrame(tfidf_detail_records)

print("✅ TF-IDF selesai dihitung")
print("Total bobot TF-IDF:", len(tfidf_records))

tfidf_df.head(10)

✅ TF-IDF selesai dihitung
Total bobot TF-IDF: 4132


,doc_id,category,title,term,term_count,total_terms,tf,df,idf,tfidf_score
0,9,machine learning,Open-environment machine learning,open,4,25,0.16,9,1.346787,0.215486
1,9,machine learning,Open-environment machine learning,environment,1,25,0.04,4,1.698970,0.067959
2,9,machine learning,Open-environment machine learning,machine,3,25,0.12,54,0.568636,0.068236
3,9,machine learning,Open-environment machine learning,learning,4,25,0.16,69,0.462181,0.073949
4,9,machine learning,Open-environment machine learning,ml,1,25,0.04,15,1.124939,0.044998
5,9,machine learning,Open-environment machine learning,short,1,25,0.04,3,1.823909,0.072956
6,9,machine learning,Open-environment machine learning,note,1,25,0.04,1,2.301030,0.092041
7,9,machine learning,Open-environment machine learning,name,1,25,0.04,2,2.000000,0.080000
8,9,machine learning,Open-environment machine learning,world,1,25,0.04,3,1.823909,0.072956
9,9,machine learning,Open-environment machine learning,used,1,25,0.04,16,1.096910,0.043876


In [25]:
# =========================================================
# CELL 9 - BENTUK MATRIX VSM
# Baris = artikel
# Kolom = term
# Nilai = bobot TF-IDF
# =========================================================
doc_ids = df["id"].astype(int).tolist()
term_to_index = {term: idx for idx, term in enumerate(terms)}
doc_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}

rows = []
cols = []
values = []

for record in tfidf_detail_records:
    rows.append(doc_to_index[record["doc_id"]])
    cols.append(term_to_index[record["term"]])
    values.append(record["tfidf_score"])

tfidf_matrix = sp.csr_matrix(
    (values, (rows, cols)),
    shape=(len(doc_ids), len(terms))
)

print("✅ Matrix VSM selesai dibuat")
print("Ukuran matrix:", tfidf_matrix.shape)
print("Total nilai non-zero:", tfidf_matrix.nnz)

✅ Matrix VSM selesai dibuat
Ukuran matrix: (200, 1723)
Total nilai non-zero: 4132


In [26]:
# =========================================================
# CELL 10 - SIMPAN FILE LOKAL TF-IDF + VSM
# =========================================================
save_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "tfidf"))
os.makedirs(save_dir, exist_ok=True)

tfidf_detail_path = os.path.join(save_dir, "tfidf_weights_detail.csv")
tfidf_weights_path = os.path.join(save_dir, "tfidf_weights.csv")
documents_path = os.path.join(save_dir, "tfidf_documents.csv")
matrix_path = os.path.join(save_dir, "tfidf_matrix.npz")
terms_path = os.path.join(save_dir, "tfidf_terms.json")
doc_ids_path = os.path.join(save_dir, "tfidf_doc_ids.json")
idf_path = os.path.join(save_dir, "idf_scores.json")

tfidf_df.to_csv(tfidf_detail_path, index=False)
pd.DataFrame(tfidf_records).to_csv(tfidf_weights_path, index=False)
df[
    [
        "id",
        "category",
        "title",
        "abstract",
        "title_tokens",
        "abstract_tokens",
        "document_tokens",
        "document_text"
    ]
].to_csv(documents_path, index=False)

sp.save_npz(matrix_path, tfidf_matrix)

with open(terms_path, "w", encoding="utf-8") as file:
    json.dump(terms, file, ensure_ascii=False, indent=2)

with open(doc_ids_path, "w", encoding="utf-8") as file:
    json.dump(doc_ids, file, ensure_ascii=False, indent=2)

with open(idf_path, "w", encoding="utf-8") as file:
    json.dump(idf_scores, file, ensure_ascii=False, indent=2)

print("✅ File lokal TF-IDF tersimpan di:", save_dir)

✅ File lokal TF-IDF tersimpan di: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf


In [27]:
# =========================================================
# CELL 11 - SIMULASI MANUAL TITLE DAN ABSTRACT TERPISAH
# Untuk penjelasan laporan/proposal
# Tidak wajib masuk ke tabel tfidf_weights
# =========================================================
manual_records = []

for _, row in df.iterrows():
    doc_id = int(row["id"])
    title = row["title"]
    category = row["category"]

    parts = {
        "title": row["title_tokens"],
        "abstract": row["abstract_tokens"]
    }

    for document_part, tokens in parts.items():
        total_terms = len(tokens)

        if total_terms == 0:
            continue

        term_counts = Counter(tokens)

        for term, count in term_counts.items():
            tf = count / total_terms
            idf = idf_scores.get(term, 0)
            tfidf_score = tf * idf

            manual_records.append({
                "doc_id": doc_id,
                "category": category,
                "title": title,
                "document_part": document_part,
                "term": term,
                "term_count": count,
                "total_terms": total_terms,
                "tf": tf,
                "df": document_frequency.get(term, 0),
                "idf": idf,
                "tfidf_score": tfidf_score
            })

manual_df = pd.DataFrame(manual_records)

manual_path = os.path.join(save_dir, "tfidf_manual_title_abstract.csv")
manual_df.to_csv(manual_path, index=False)

print("✅ Simulasi manual title/abstract tersimpan:", manual_path)
manual_df.head(10)

✅ Simulasi manual title/abstract tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\tfidf_manual_title_abstract.csv


,doc_id,category,title,document_part,term,term_count,total_terms,tf,df,idf,tfidf_score
0,9,machine learning,Open-environment machine learning,title,open,1,4,0.250000,9,1.346787,0.336697
1,9,machine learning,Open-environment machine learning,title,environment,1,4,0.250000,4,1.698970,0.424743
2,9,machine learning,Open-environment machine learning,title,machine,1,4,0.250000,54,0.568636,0.142159
3,9,machine learning,Open-environment machine learning,title,learning,1,4,0.250000,69,0.462181,0.115545
4,9,machine learning,Open-environment machine learning,abstract,open,3,21,0.142857,9,1.346787,0.192398
5,9,machine learning,Open-environment machine learning,abstract,learning,3,21,0.142857,69,0.462181,0.066026
6,9,machine learning,Open-environment machine learning,abstract,ml,1,21,0.047619,15,1.124939,0.053569
7,9,machine learning,Open-environment machine learning,abstract,short,1,21,0.047619,3,1.823909,0.086853
8,9,machine learning,Open-environment machine learning,abstract,note,1,21,0.047619,1,2.301030,0.109573
9,9,machine learning,Open-environment machine learning,abstract,name,1,21,0.047619,2,2.000000,0.095238


In [28]:
# =========================================================
# CELL 12 - CONTOH VSM MANUAL UNTUK 1 ARTIKEL
# =========================================================
example_doc_id = int(df.iloc[0]["id"])

example_terms = (
    tfidf_df[tfidf_df["doc_id"] == example_doc_id]
    .sort_values("tfidf_score", ascending=False)
    .head(5)["term"]
    .tolist()
)

example_vsm = manual_df[
    (manual_df["doc_id"] == example_doc_id) &
    (manual_df["term"].isin(example_terms))
].pivot_table(
    index=["doc_id", "document_part"],
    columns="term",
    values="tfidf_score",
    fill_value=0
).reset_index()

print("Contoh artikel ID:", example_doc_id)
print("5 term utama:", example_terms)

example_vsm

Contoh artikel ID: 9
5 term utama: ['open', 'refer', 'ood', 'distribution', 'outof']


term,doc_id,document_part,distribution,ood,open,outof,refer
0,9,abstract,0.109573,0.109573,0.192398,0.109573,0.109573
1,9,title,0.000000,0.000000,0.336697,0.000000,0.000000


In [29]:
# =========================================================
# CELL 13 - HELPER INSERT SUPABASE
# =========================================================
def insert_batches(table_name, records, batch_size=500):
    total = len(records)

    if total == 0:
        print(f"⚠️ Tidak ada data untuk dimasukkan ke {table_name}")
        return

    for start in range(0, total, batch_size):
        batch = records[start:start + batch_size]

        supabase.table(table_name).insert(batch).execute()

        print(f"  → Batch {start // batch_size + 1}: {len(batch)} data")

    print(f"✅ Insert {total} data ke {table_name}")

In [30]:
# =========================================================
# CELL 14 - INSERT TF-IDF KE SUPABASE
# =========================================================
insert_batches(
    table_name=TFIDF_TABLE,
    records=tfidf_records,
    batch_size=500
)

check_tfidf = (
    supabase.table(TFIDF_TABLE)
    .select("doc_id", count="exact")
    .limit(1)
    .execute()
)

print("✅ Jumlah data di tfidf_weights:", check_tfidf.count)

APIError: {'message': 'duplicate key value violates unique constraint "tfidf_weights_pkey"', 'code': '23505', 'hint': None, 'details': 'Key (doc_id, term)=(9, open) already exists.'}

In [ ]:
# =========================================================
# CELL 15 - OUTPUT VSM READABLE
# Hati-hati: bisa besar, tapi berguna untuk inspeksi
# =========================================================
vsm_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=doc_ids,
    columns=terms
)

vsm_df.index.name = "doc_id"

vsm_readable_path = os.path.join(save_dir, "vsm_matrix_readable.csv")
vsm_df.to_csv(vsm_readable_path)

print("✅ VSM readable tersimpan:", vsm_readable_path)
print("Ukuran VSM:", vsm_df.shape)

vsm_df.head()

✅ VSM readable tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\vsm_matrix_readable.csv
Ukuran VSM: (200, 1723)


,000,10,100m,12,13,16,19,1959,1k,20,...,works,world,would,xss,years,you,young,your,youth,zone
doc_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# =========================================================
# CELL 16 - SAMPLE VSM PER KATEGORI
# Ini pengganti sample lama yang cuma head(10)
# =========================================================
sample_ids = []

for category, group in df.groupby("category"):
    sample_ids.extend(group["id"].head(3).astype(int).tolist())

sample_indices = [
    doc_ids.index(doc_id)
    for doc_id in sample_ids
    if doc_id in doc_ids
]

sample_matrix = tfidf_matrix[sample_indices]
sample_doc_ids = [doc_ids[index] for index in sample_indices]

top_terms_sample = (
    pd.DataFrame({
        "term": terms,
        "total_weight": sample_matrix.sum(axis=0).A1
    })
    .query("total_weight > 0")
    .sort_values("total_weight", ascending=False)
    .head(15)["term"]
    .tolist()
)

vsm_sample_df = pd.DataFrame(
    sample_matrix.toarray(),
    index=sample_doc_ids,
    columns=terms
)

vsm_sample_df = vsm_sample_df[top_terms_sample]
vsm_sample_df.index.name = "doc_id"

category_lookup = df.set_index("id")["category"].to_dict()
vsm_sample_df.insert(
    0,
    "category",
    [category_lookup.get(doc_id, "") for doc_id in sample_doc_ids]
)

vsm_sample_path = os.path.join(save_dir, "vsm_matrix_sample.csv")
vsm_sample_df.to_csv(vsm_sample_path)

print("✅ Sample VSM per kategori tersimpan:", vsm_sample_path)
print("Term yang ditampilkan:", top_terms_sample)

vsm_sample_df

✅ Sample VSM per kategori tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\vsm_matrix_sample.csv
Term yang ditampilkan: ['cyber', 'their', 'security', 'web', 'advertising', 'learning', 'how', 'mobile', 'via', 'microbiologists', 'applications', 'brand', 'machine', 'domain', 'future']


,category,cyber,their,security,web,advertising,learning,how,mobile,via,microbiologists,applications,brand,machine,domain,future
doc_id,,,,,,,,,,,,,,,,
142,cyber security,0.061979,0.051114,0.028033,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000
101,cyber security,0.103298,0.000000,0.093445,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.191752,0.000000
102,cyber security,0.095352,0.000000,0.064692,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.096895
1,machine learning,0.000000,0.000000,0.000000,0.000000,0.000000,0.080379,0.100426,0.000000,0.000000,0.20009,0.000000,0.000000,0.098893,0.000000,0.000000
2,machine learning,0.000000,0.000000,0.000000,0.000000,0.000000,0.115545,0.000000,0.000000,0.063453,0.00000,0.000000,0.000000,0.047386,0.000000,0.000000
3,machine learning,0.000000,0.040891,0.000000,0.000000,0.000000,0.036974,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.045491,0.000000,0.000000
70,mobile application,0.000000,0.000000,0.000000,0.000000,0.242424,0.000000,0.034997,0.071935,0.092296,0.00000,0.000000,0.121212,0.000000,0.000000,0.000000
51,mobile application,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.042774,0.043960,0.056403,0.00000,0.000000,0.074074,0.000000,0.000000,0.046653
52,mobile application,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.048121,0.098910,0.000000,0.00000,0.043573,0.000000,0.000000,0.000000,0.000000


In [ ]:
# =========================================================
# CELL 17 - TOP TERM PER KATEGORI
# Untuk membuktikan 4 kategori punya bobot masing-masing
# =========================================================
top_category_rows = []

for category, group in df.groupby("category"):
    category_doc_ids = group["id"].astype(int).tolist()
    category_indices = [
        doc_ids.index(doc_id)
        for doc_id in category_doc_ids
        if doc_id in doc_ids
    ]

    if not category_indices:
        continue

    category_matrix = tfidf_matrix[category_indices]

    category_terms_df = pd.DataFrame({
        "category": category,
        "term": terms,
        "total_weight": category_matrix.sum(axis=0).A1,
        "document_count": (category_matrix > 0).sum(axis=0).A1
    })

    category_terms_df = (
        category_terms_df[category_terms_df["total_weight"] > 0]
        .sort_values("total_weight", ascending=False)
        .head(10)
    )

    top_category_rows.append(category_terms_df)

top_terms_per_category_df = pd.concat(top_category_rows, ignore_index=True)

top_terms_per_category_path = os.path.join(save_dir, "top_terms_per_category.csv")
top_terms_per_category_df.to_csv(top_terms_per_category_path, index=False)

print("✅ Top term per kategori tersimpan:", top_terms_per_category_path)

top_terms_per_category_df

✅ Top term per kategori tersimpan: d:\Tugas Akhir\paperci_artikel\backend\data\tfidf\top_terms_per_category.csv


,category,term,total_weight,document_count
0,cyber security,cyber,4.108037,48
1,cyber security,security,3.431982,46
2,cyber security,cybersecurity,1.594846,17
3,cyber security,threats,0.669210,12
4,cyber security,attacks,0.560000,11
5,cyber security,incidents,0.507074,4
6,cyber security,challenges,0.504226,12
7,cyber security,systems,0.423270,6
8,cyber security,risk,0.407876,3
9,cyber security,review,0.371279,13


In [ ]:
# =========================================================
# CELL 18 - VALIDASI TERM KATEGORI UTAMA
# =========================================================
category_terms = [
    "machine", "learning",
    "web", "develop", "development",
    "cyber", "security", "secure",
    "mobile", "application", "app", "apps"
]

available_terms = [term for term in category_terms if term in terms]
missing_terms = [term for term in category_terms if term not in terms]

print("Term tersedia:", available_terms)
print("Term tidak ada:", missing_terms)

term_summary_df = pd.DataFrame({
    "term": available_terms,
    "document_count": [
        int((tfidf_matrix[:, terms.index(term)] > 0).sum())
        for term in available_terms
    ],
    "total_weight": [
        float(tfidf_matrix[:, terms.index(term)].sum())
        for term in available_terms
    ]
}).sort_values("total_weight", ascending=False)

term_summary_path = os.path.join(save_dir, "category_term_summary.csv")
term_summary_df.to_csv(term_summary_path, index=False)

term_summary_df

Term tersedia: ['machine', 'learning', 'web', 'develop', 'development', 'cyber', 'security', 'secure', 'mobile', 'application', 'app', 'apps']
Term tidak ada: []


,term,document_count,total_weight
5,cyber,48,4.108037
1,learning,69,3.979429
0,machine,54,3.865188
6,security,55,3.721660
8,mobile,51,3.303019
2,web,50,3.107354
9,application,63,2.153161
11,apps,37,2.038807
10,app,34,1.638423
4,development,13,0.626006
